# Drive 데이터에서 헤드 분석 전체 재실험

`Drive데이터_실험입력준비_Colab.ipynb`가 만든 `drive_study_inputs_v1.json` 하나를 업로드하고 Colab GPU에서 위에서 아래로 실행하세요. 기존 CSV·노트북은 변경하지 않으며 새 결과는 `drive_head_*.csv`로 브라우저에 다운로드합니다. 모델은 Qwen2.5-1.5B-Instruct **FP16**이고 보정·양자화 모델은 이 노트북에서 쓰지 않습니다.

1. `validation` 1번 묶음의 512토큰으로 전체 레이어를 하나씩 끄고 상위 6개 레이어의 헤드 72개를 개별 탐색합니다.
2. 같은 `validation` 1번 묶음의 2048토큰에서 모든 Query 헤드 336개의 **국소 손실 기울기**를 계산합니다. 전체 상위 4개와 1단계에서 개별 평가하지 않은 레이어의 상위 4개를 고릅니다.
3. 두 방식의 후보와 **기존에 발견한 0·10, 0·8, 0·11, 23·8**, 대조용 19·6을 합쳐 중복 없이 `test` 3개 묶음(각 8192토큰)에서 헤드를 실제로 끕니다. test의 결과로 후보를 다시 고르지 않습니다.

`S = (마스킹 후 한국어 PPL / 원본 한국어 PPL − 1) × 100 − (마스킹 후 영어 PPL / 원본 영어 PPL − 1) × 100`입니다. 기울기 차이는 이 S와 단위가 다르고 후보 탐색에만 사용합니다. Drive의 영어 원문은 기존 WikiText 계열과 겹칠 수 있으며 준비 노트북의 정확한 문장 중복 제거가 근사·부분 중복까지 보장하지 않습니다. GPU 시간이 오래 걸릴 수 있습니다.


In [ ]:
import csv
import hashlib
import json
import math
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import torch
import transformers
from google.colab import files
from IPython.display import display
from transformers import AutoModelForCausalLM

if not torch.cuda.is_available():
    raise RuntimeError("Colab GPU 런타임을 사용하세요.")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("drive_study_inputs_v1.json 하나만 업로드하세요.")
input_bytes = next(iter(uploaded.values()))
input_sha256 = hashlib.sha256(input_bytes).hexdigest()
study = json.loads(input_bytes.decode("utf-8"))
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
if study.get("format") != "team22_drive_study_v1" or study.get("model_id") != MODEL_ID:
    raise ValueError("입력 JSON의 형식 또는 모델이 다릅니다.")
for split in ("validation", "test"):
    for lang in ("ko", "en"):
        groups = study[split][lang]
        if len(groups) != 3 or any(len(group) != 8192 for group in groups):
            raise ValueError(f"{split}/{lang}: 8192토큰 묶음 3개가 필요합니다.")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16
).to("cuda:0").eval()
model.requires_grad_(False)
model.config.use_cache = False
N_LAYERS = len(model.model.layers)
N_HEADS = model.config.num_attention_heads
HEAD_DIM = getattr(model.config, "head_dim", None) or model.config.hidden_size // N_HEADS
OUTPUT_DIR = Path("/content/team22_drive_retest/head")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0), "입력 SHA-256:", input_sha256)
print("레이어:", N_LAYERS, "Query 헤드/레이어:", N_HEADS)


In [ ]:
def mean_nll(token_ids, layer=None, head=None):
    hook = None
    if layer is not None:
        projection = model.model.layers[layer].self_attn.o_proj

        def mask(_module, args):
            x = args[0].clone()
            if x.shape[-1] != N_HEADS * HEAD_DIM:
                raise ValueError(f"예상과 다른 attention 폭: {x.shape[-1]}")
            if head is None:
                x.zero_()
            else:
                start = head * HEAD_DIM
                x[..., start:start + HEAD_DIM] = 0
            return (x,)

        hook = projection.register_forward_pre_hook(mask)
    try:
        total_nll, predicted = 0.0, 0
        for start in range(0, len(token_ids), 512):
            chunk = token_ids[start:start + 512]
            if len(chunk) != 512:
                raise ValueError("토큰 길이가 512의 배수가 아닙니다.")
            input_ids = torch.tensor([chunk], dtype=torch.long, device="cuda:0")
            with torch.inference_mode():
                output = model(input_ids=input_ids, labels=input_ids, use_cache=False)
            total_nll += output.loss.item() * 511
            predicted += 511
            del input_ids, output
        return total_nll / predicted
    finally:
        if hook is not None:
            hook.remove()

def ppl_change(tokens, baseline_nll, layer, head=None):
    return 100 * math.expm1(mean_nll(tokens, layer, head) - baseline_nll)

def write_csv(name, rows):
    path = OUTPUT_DIR / name
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    return path

validation_short = {lang: study["validation"][lang][0][:512] for lang in ("ko", "en")}
base_short = {lang: mean_nll(tokens) for lang, tokens in validation_short.items()}
assert abs(mean_nll(validation_short["ko"]) - base_short["ko"]) < 1e-3


## 1. 레이어 → 개별 헤드 마스킹 탐색 (validation)

각 Query 헤드를 끌 때 `o_proj`에 들어가는 해당 헤드 출력 구간만 0으로 만듭니다. 레이어 전체 마스킹 점수는 개별 헤드 점수와 비교하거나 합산하지 않습니다.


In [ ]:
layer_rows = []
for layer in range(N_LAYERS):
    ko = ppl_change(validation_short["ko"], base_short["ko"], layer)
    en = ppl_change(validation_short["en"], base_short["en"], layer)
    layer_rows.append({"layer": layer, "ko_change_pct": ko,
                       "en_change_pct": en, "S_pp": ko - en})
layer_candidates = sorted(
    (row for row in layer_rows if row["ko_change_pct"] > 0),
    key=lambda row: row["S_pp"], reverse=True
)[:6]
if len(layer_candidates) != 6:
    raise RuntimeError("한국어 PPL이 증가한 레이어 후보 6개가 필요합니다.")
print("헤드 탐색 레이어:", [row["layer"] for row in layer_candidates])
write_csv("drive_head_layer_screen.csv", layer_rows)


In [ ]:
head_rows = []
for layer_row in layer_candidates:
    layer = layer_row["layer"]
    for head in range(N_HEADS):
        ko = ppl_change(validation_short["ko"], base_short["ko"], layer, head)
        en = ppl_change(validation_short["en"], base_short["en"], layer, head)
        head_rows.append({"layer": layer, "head": head,
                          "ko_change_pct": ko, "en_change_pct": en,
                          "S_pp": ko - en})
mask_candidates = sorted(
    (row for row in head_rows if row["ko_change_pct"] > 0),
    key=lambda row: row["S_pp"], reverse=True
)[:10]
if len(mask_candidates) != 10:
    raise RuntimeError("한국어 PPL이 증가한 헤드 후보 10개가 필요합니다.")
print("마스킹 상위 후보:", [(r["layer"], r["head"]) for r in mask_candidates])
write_csv("drive_head_mask_screen.csv", head_rows)


## 2. 모든 헤드의 게이트 기울기 탐색 (validation)

모든 헤드 출력 게이트가 1인 상태에서 평균 NLL을 게이트로 미분합니다. 모델 가중치는 고정합니다. 기울기가 유한하지 않으면 결과를 사용하지 않습니다.


In [ ]:
def gate_gradients(token_ids):
    gates = torch.ones(N_LAYERS, N_HEADS, device="cuda:0", requires_grad=True)
    hooks = []
    for layer in range(N_LAYERS):
        projection = model.model.layers[layer].self_attn.o_proj

        def gate_output(_module, args, layer=layer):
            x = args[0]
            if x.shape[-1] != N_HEADS * HEAD_DIM:
                raise ValueError(f"예상과 다른 attention 폭: {x.shape[-1]}")
            heads = x.reshape(*x.shape[:-1], N_HEADS, HEAD_DIM)
            scaled = heads * gates[layer].to(x.dtype).view(1, 1, N_HEADS, 1)
            return (scaled.reshape_as(x),)

        hooks.append(projection.register_forward_pre_hook(gate_output))
    try:
        total = torch.zeros_like(gates)
        first_loss = None
        for start in range(0, len(token_ids), 512):
            input_ids = torch.tensor([token_ids[start:start + 512]],
                                     dtype=torch.long, device="cuda:0")
            output = model(input_ids=input_ids, labels=input_ids, use_cache=False)
            if first_loss is None:
                first_loss = output.loss.item()
            grad = torch.autograd.grad(output.loss, gates)[0]
            if not torch.isfinite(grad).all():
                raise RuntimeError("게이트 기울기에 NaN/Inf가 있습니다.")
            total += grad.detach()
            del input_ids, output, grad
        assert total.abs().sum().item() > 0
        return total / (len(token_ids) // 512), first_loss
    finally:
        for hook in hooks:
            hook.remove()

validation_long = {lang: study["validation"][lang][0][:2048]
                   for lang in ("ko", "en")}
grad_ko, gated_loss = gate_gradients(validation_long["ko"])
assert abs(gated_loss - mean_nll(validation_long["ko"][:512])) < 1e-3
grad_en, _ = gate_gradients(validation_long["en"])

gradient_rows = []
selected_layers = {row["layer"] for row in layer_candidates}
for layer in range(N_LAYERS):
    for head in range(N_HEADS):
        ko, en = -grad_ko[layer, head].item(), -grad_en[layer, head].item()
        gradient_rows.append({
            "layer": layer, "head": head,
            "ko_local_nll_increase": ko, "en_local_nll_increase": en,
            "S_gradient": ko - en,
            "mask_screened_layer": layer in selected_layers,
        })
ranked = sorted(
    (row for row in gradient_rows if row["ko_local_nll_increase"] > 0),
    key=lambda row: row["S_gradient"], reverse=True
)
gradient_candidates = {}
for row in ranked[:4]:
    gradient_candidates[row["layer"], row["head"]] = "gradient_global"
for row in (row for row in ranked if not row["mask_screened_layer"]):
    if sum(v == "gradient_unseen_layer" for v in gradient_candidates.values()) >= 4:
        break
    gradient_candidates.setdefault((row["layer"], row["head"]),
                                   "gradient_unseen_layer")
if len(gradient_candidates) != 8:
    raise RuntimeError("기울기 후보 8개를 선정하지 못했습니다.")
print("기울기 후보:", gradient_candidates)
write_csv("drive_head_gradient_screen.csv", gradient_rows)


## 3. 고정 후보와 새 후보를 test에서 검증

새 test 데이터를 보고 후보를 추가하거나 순위를 다시 정하지 않습니다. 기존 연구 후보와 대조 후보도 함께 평가합니다. 같은 test 토큰을 모든 후보가 공유합니다.


In [ ]:
reasons = defaultdict(set)
for row in mask_candidates:
    reasons[row["layer"], row["head"]].add("mask_top10")
for key, reason in gradient_candidates.items():
    reasons[key].add(reason)
for key in ((0, 10), (0, 8), (0, 11), (23, 8)):
    reasons[key].add("previous_candidate")
reasons[19, 6].add("previous_control")
if any(layer >= N_LAYERS or head >= N_HEADS for layer, head in reasons):
    raise ValueError("기존 후보의 레이어·헤드 번호가 모델 구조를 벗어납니다.")

baseline = {(lang, group): mean_nll(study["test"][lang][group - 1])
            for lang in ("ko", "en") for group in (1, 2, 3)}
validation_rows = []
for layer, head in sorted(reasons):
    selection = "+".join(sorted(reasons[layer, head]))
    for group in (1, 2, 3):
        ko = ppl_change(study["test"]["ko"][group - 1], baseline["ko", group],
                        layer, head)
        en = ppl_change(study["test"]["en"][group - 1], baseline["en", group],
                        layer, head)
        validation_rows.append({
            "layer": layer, "head": head, "selection": selection, "group": group,
            "baseline_ko_ppl": math.exp(baseline["ko", group]),
            "baseline_en_ppl": math.exp(baseline["en", group]),
            "ko_change_pct": ko, "en_change_pct": en, "S_pp": ko - en,
        })
    write_csv("drive_head_test_progress.csv", validation_rows)
    print("test 완료:", layer, head)

summary_rows = []
for layer, head in sorted(reasons):
    pair = [r for r in validation_rows if (r["layer"], r["head"]) == (layer, head)]
    summary_rows.append({
        "layer": layer, "head": head,
        "selection": "+".join(sorted(reasons[layer, head])),
        "mean_ko_change_pct": sum(r["ko_change_pct"] for r in pair) / 3,
        "mean_en_change_pct": sum(r["en_change_pct"] for r in pair) / 3,
        "mean_S_pp": sum(r["S_pp"] for r in pair) / 3,
        "positive_ko_groups": sum(r["ko_change_pct"] > 0 for r in pair),
        "positive_S_groups": sum(r["S_pp"] > 0 for r in pair),
    })
summary_rows.sort(key=lambda r: r["mean_S_pp"], reverse=True)
by_group_path = write_csv("drive_head_test_by_group.csv", validation_rows)
summary_path = write_csv("drive_head_test_summary.csv", summary_rows)
run_info = {
    "model_id": MODEL_ID, "input_sha256": input_sha256,
    "source_sha256": study["source_sha256"],
    "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
    "transformers": transformers.__version__,
    "selection_split": "validation", "test_split": "test",
    "test_groups_per_language": 3, "test_tokens_per_group": 8192,
    "mask_selected_layers": sorted(selected_layers),
    "candidate_count": len(reasons),
    "generated_utc": datetime.now(timezone.utc).isoformat(),
}
info_path = OUTPUT_DIR / "drive_head_runtime.json"
info_path.write_text(json.dumps(run_info,ensure_ascii=False,indent=2),encoding="utf-8")
print("검증 후보:", len(reasons), "묶음별 행:", len(validation_rows))
display(summary_rows)


## 4. 결과 CSV를 브라우저에서 하나씩 다운로드

생성된 파일 6개를 각각 다른 셀에서 내려받습니다. 여러 다운로드를 한 셀에 묶으면 브라우저가 일부를 막을 수 있습니다. 다운로드 완료를 확인한 뒤 런타임을 종료하세요.


In [ ]:
files.download(str(OUTPUT_DIR / "drive_head_layer_screen.csv"))


In [ ]:
files.download(str(OUTPUT_DIR / "drive_head_mask_screen.csv"))


In [ ]:
files.download(str(OUTPUT_DIR / "drive_head_gradient_screen.csv"))


In [ ]:
files.download(str(by_group_path))


In [ ]:
files.download(str(summary_path))


In [ ]:
files.download(str(info_path))
